In [79]:
#python-spatial-2025-10-22
# plotting
import matplotlib.pyplot as plt
# data wrangling
import pandas as pd
# web-based base maps
import folium

In [80]:
# optimized for Google Colab
# tested in VS Code locally as well, some pip / conda installs
#  may be necessary

In [81]:
# option return to run line and go to next

In [82]:
lat = 43.7031000
lon = -72.28854

In [85]:
# the 'folium' library makes mapping in Python very easy!
map = folium.Map(location=[lat, lon], zoom_start=15)

# un-comment to show the map
#map

In [ ]:
#pip install geopandas

In [86]:
# importing a few more libraries
# geospatial data wrangling & geometric analysis
import geopandas as gpd
# working with shapes and spatial objects
import shapely

In [87]:
point_csv = pd.read_csv('https://rcweb.dartmouth.edu/homes/f002d69/workshops/data/bear-sightings.csv')
#point_csv.head(3)

In [90]:
# Specify the path to your .shp file
shapefile_path = "https://rcweb.dartmouth.edu/homes/f002d69/workshops/data/nationalparks_ak.zip"

# Read the shapefile into a GeoDataFrame
polygons_gdf = gpd.read_file(shapefile_path)
# Display the first few rows of the GeoDataFrame
#print(polygons_gdf.head())

In [91]:
# convert the CSV, which has latitude and longitude values in it, to a geopandas spatial object
points = gpd.GeoDataFrame(point_csv, geometry=gpd.points_from_xy(point_csv.longitude, point_csv.latitude))
#points.head(3)
# note the 'geometry' field

In [94]:
# A 'crs' is a Coordinate Reference System
# https://epsg.io/4326 is a global project system
# set the coodinate reference system of the points to epsg 4326
points.crs='EPSG:4326'
points = points.to_crs(polygons_gdf.crs)
#points.crs
#polygons_gdf.crs

In [95]:
#Spatial Analysis tool 'sjoin'
# https://geopandas.org/en/stable/docs/reference/api/geopandas.sjoin.html

# and https://geopandas.org/en/stable/gallery/spatial_joins.html

# do the spatial join! this line of code does a spatial overlay for points within polygons
points_in_polygons = gpd.sjoin(points, polygons_gdf, predicate='within')
#points_in_polygons = gpd.sjoin(points, polygons_gdf, how="left", predicate='within') # how saves all points

#points_in_polygons.head(3)

# note that some, not all, bears in this table have a park name associated with them

In [96]:
points_outside_polygons = gpd.sjoin(points, polygons_gdf, how='left',
                              predicate='within').query('index_right.isna()')
#points_outside_polygons.head(3)

In [97]:
# Combine the two GeoDataFrames
all_points = pd.concat([points_in_polygons, points_outside_polygons])

# Sort the combined GeoDataFrame by 'bear.id'
all_points = all_points.sort_values(by='bear.id')

# Display the first few rows of the combined GeoDataFrame
#display(all_points.head())

# Display the last few rows of the combined GeoDataFrame to see some points outside polygons
#display(all_points.tail())

In [98]:
# Combine the two GeoDataFrames
all_points = pd.concat([points_in_polygons, points_outside_polygons])

# Sort the combined GeoDataFrame by 'bear.id'
all_points = all_points.sort_values(by='bear.id')
#all_points.head(3)

In [ ]:
# make a map here:
polygons_gdf.plot(facecolor='none') # polygons, show with black outline, no fill
points.plot(color = 'blue' ,ax=plt.gca())  # points outside polygons
points_in_polygons.plot(color='red', ax=plt.gca())  # points inside polygons
plt.axis("off")
#plt.show()

In [ ]:
# make a leaflet map:

# let's make a leaflet map in Google Colab
# center the map (this time a folium / leaflet map)
mid_lat = polygons_gdf.geometry.centroid.y.mean()
mid_lon = polygons_gdf.geometry.centroid.x.mean()
# create the map / initialize the map
map_folium = folium.Map(location=[mid_lat, mid_lon], zoom_start=5)
# to clear the map, re-run this cell to reinitialize the map

# loop over the polygons, and place them in the map
# to see the popup box for a polygon, click on the polygon, it should display
# the name of the park in the popup window
for idx, row in polygons_gdf.iterrows():
    # Ensure geometry is a Polygon or MultiPolygon
    geom = row.geometry
    if geom.geom_type == 'Polygon':
        coords = [(y, x) for x, y in geom.exterior.coords]
        folium.Polygon(
            locations=coords,
            fill=True,
            popup=f"Polygon: {row['Unit_Name']}"
        ).add_to(map_folium)
    elif geom.geom_type == 'MultiPolygon':
        # If multipolygon, add each polygon separately
        for poly in geom.geoms:
            coords = [(y, x) for x, y in poly.exterior.coords]
            folium.Polygon(
                locations=coords,
                fill=True,
                popup=f"Polygon: {row['Unit_Name']}"
            ).add_to(map_folium)

            # plot all of the bear sightings...
# Iterate over the rows of your GeoDataFrame
for index, row in points.iterrows():
    # Get the coordinates of the point
    lat = row.geometry.y
    lon = row.geometry.x

    # Add a marker to the map
    folium.Marker([lat, lon], popup=row['bear.id']).add_to(map_folium) # Assuming you have a 'name' column for popups
# Display the folium map (outside the loop)
#map_folium


# To clear the map, re-initialize it
#my_map = folium.Map(location=[40.7128, -74.0060], zoom_start=9)


In [107]:
# next, plot the bears inside polygons, using red icons ,

#popup_text = f"{row['bear.id']} {row['Unit_Name']}"

for index, row in points_in_polygons.iterrows():
    # Get the coordinates of the point
    lat = row.geometry.y
    lon = row.geometry.x

    # Add a marker to the map
    folium.Marker([lat, lon], icon=folium.Icon(color="red"), popup=f"Bear ID: {row['bear.id']}, Park: {row['Unit_Name']}", color='red').add_to(map_folium) # Assuming you have a 'name' column for popups
#             icon=folium.Icon(color="red"),

#map_folium

In [108]:
# next, plot the bears inside polygons, using red icons ,

#popup_text = f"{row['bear.id']} {row['Unit_Name']}"

for index, row in points_in_polygons.iterrows():
    # Get the coordinates of the point
    lat = row.geometry.y
    lon = row.geometry.x

    # Add a marker to the map
    folium.Marker([lat, lon], icon=folium.Icon(color="red"), popup=f"{row['bear.id']}", color='red').add_to(map_folium) # Assuming you have a 'name' column for popups
#             icon=folium.Icon(color="red"),

#map_folium

In [ ]:
popup_text = f"{row['bear.id']} {row['Unit_Name']}"
folium.Marker(
    [lat, lon],
    icon=folium.Icon(color="red"),
    popup=popup_text
).add_to(map_folium)
map_folium

To export, see code cell below

In [ ]:
# to export the results to an external CSV:

# if running in local instance of python, like VS Code, Export to CSV
points_in_polygons.to_csv('bears-in-parks.csv', index=False)

# If running in Colab / Jupyter you can download the file with:
from google.colab import files
files.download('bears-in-parks.csv')

Heat map example"

In [ ]:
import folium
from folium.plugins import HeatMap
import pandas as pd
import numpy as np

# 1. Generate sample point data (e.g., crime incidents, points of interest)
# In a real-world scenario, this data would come from a CSV, database, etc.
np.random.seed(42) # for reproducibility
num_points = 200
data = {
    'latitude': 37.7749 + (np.random.rand(num_points) - 0.5) * 0.1, # Centered around San Francisco
    'longitude': -122.4194 + (np.random.rand(num_points) - 0.5) * 0.1,
    'intensity': np.random.rand(num_points) * 10 # Example intensity for heatmap
}
df = pd.DataFrame(data)

# 2. Create a base Folium map
# Center the map around the average latitude and longitude of the data
map_center = [df['latitude'].mean(), df['longitude'].mean()]
m = folium.Map(location=map_center, zoom_start=12, tiles='OpenStreetMap')

# Create feature groups for layering
marker_layer = folium.FeatureGroup(name='Markers').add_to(m)
heatmap_layer = folium.FeatureGroup(name='Heatmap').add_to(m)


# 3. Add individual point markers to the marker layer
for index, row in df.iterrows():
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        popup=f"Point {index+1}", # Add a popup with information
        icon=folium.Icon(color='blue', icon='info-sign') # Customize marker icon
    ).add_to(marker_layer)

# 4. Add a HeatMap layer to the heatmap layer
# The HeatMap requires a list of lists, where each inner list is [latitude, longitude, intensity]
heat_data = [[row['latitude'], row['longitude'], row['intensity']] for index, row in df.iterrows()]
HeatMap(heat_data).add_to(heatmap_layer)

# Add layer control to the map
folium.LayerControl().add_to(m)

m